# Introduction to Model Context Protocol — Course NotesNotes from [**Introduction to MCP**](https://anthropic-partners.skilljar.com/introduction-to-model-context-protocol)(Anthropic Partner Academy), worked through against the project in[`Examples/`](Examples/).**What the course builds:** a document-management MCP server exposing all three MCPprimitives, plus a CLI client that wires that server into a Claude conversation with`@`-mentions and `/`-commands.| Module | Lessons | Notebook section ||---|---|---|| Introduction | Welcome · Introducing MCP · MCP clients | [1](#1--introduction) || Hands-on with MCP servers | Project setup · Defining tools · Server inspector | [2](#2--hands-on-with-mcp-servers) || Connecting with MCP clients | Implementing a client · Defining/Accessing resources · Defining prompts · Prompts in the client | [3](#3--connecting-with-mcp-clients) || Assessment and wrap up | Final assessment · MCP review | [4](#4--assessment-and-wrap-up) |**Prerequisites:** Python, plus passing familiarity with JSON and request/response HTTP patterns.---## Running the code cellsThe runnable cells build a small FastMCP server **inside this notebook** and introspect itdirectly — no subprocess, no Inspector, no API key. That works because `FastMCP` exposes thesame `list_tools` / `call_tool` / `read_resource` / `get_prompt` methods that the protocolhandlers call, so you can exercise the server logic without a transport in between.Select the kernel at `Examples/.venv` (`mcp` 1.26.0, `anthropic` 0.84.0). Cells use top-level`await`, which Jupyter supports natively.Anything that genuinely needs a live client-server pair is shown as a commented excerpt fromthe working files instead, since it can't complete inside a notebook cell.

---# 1 — Introduction## 1.1 Welcome to the courseThe through-line of the course is one project, built in layers:1. A **server** (`mcp_server.py`) that owns a set of documents and exposes tools, resources   and prompts over them.2. A **client** (`mcp_client.py`) that launches that server and speaks the protocol to it.3. A **chat layer** (`core/`) that puts Claude in the middle — the model calls the server's   tools, and the CLI surfaces its resources and prompts to the human.Everything runs locally over stdio, so there's no deployment step to worry about.

## 1.2 Introducing MCP### The problem it solvesBefore MCP, every (AI application × external system) pair needed its own bespoke integration.*M* applications and *N* systems means **M×N** integrations, each one reimplementing the same"describe a capability, invoke it, return the result" logic.MCP is an open protocol that standardises that interface, turning the problem into **M+N**:each application implements a client once, each system implements a server once, and anyclient can talk to any server. The usual comparison is USB-C — one connector instead of adrawer of proprietary cables. LSP is the closer technical analogy: it did exactly this foreditors and language tooling.### Architecture```┌─────────────────────────── HOST ────────────────────────────┐│  (Claude Desktop, Claude Code, your CLI app, the Inspector) ││                                                             ││   ┌──────────┐        ┌──────────┐        ┌──────────┐      ││   │ Client 1 │        │ Client 2 │        │ Client 3 │      ││   └────┬─────┘        └────┬─────┘        └────┬─────┘      │└────────┼───────────────────┼───────────────────┼────────────┘         │ 1:1               │ 1:1               │ 1:1    ┌────┴─────┐        ┌────┴─────┐        ┌────┴─────┐    │ Server A │        │ Server B │        │ Server C │    │  (docs)  │        │  (git)   │        │  (db)    │    └──────────┘        └──────────┘        └──────────┘```- **Host** — the application the user actually interacts with. It holds the conversation and  the model.- **Client** — the protocol-speaking object inside the host. The relationship is strictly  **one client per server**; connecting to three servers means three client instances. That  is why `core/chat.py` carries a `clients: dict[str, MCPClient]` rather than a single client.- **Server** — the program exposing capabilities. It is passive: it never initiates anything,  it answers.A useful framing from the project's own header comment: a server is a vending machine. Stockedand waiting, but nothing happens until a customer walks up.### The layers**Transport** — how bytes move.| Transport | Where the server runs | Used by ||---|---|---|| **stdio** | Local subprocess of the host | This course; most local integrations || **Streamable HTTP** | Remote, over HTTP(S) | Hosted/shared servers |**Data layer** — [JSON-RPC 2.0](https://www.jsonrpc.org/specification) for every message. Withstdio, those JSON-RPC frames are literally written to the child process's stdin and read backoff its stdout, which is why a server must never `print()` to stdout — it would corrupt thestream. (Note `log_level="ERROR"` on the `FastMCP` constructor in `mcp_server.py`.)### The three primitivesThis is the part worth memorising, because the whole course is organised around it:| Primitive | Controlled by | Analogous to | Who decides to use it ||---|---|---|---|| **Tools** | Model | `POST` endpoint | Claude picks one mid-conversation || **Resources** | Application | `GET` endpoint | The client fetches by URI || **Prompts** | User | Slash command / template | The human triggers it deliberately |Tools are for the AI, resources are for the client, prompts are for the human. Servers can also*consume* from the host — **sampling** (ask the host's model for a completion), **roots**(filesystem scope), **elicitation** (ask the user a question) — but the course focuses on thethree above.

In [ ]:
# The exact JSON-RPC 2.0 envelopes MCP uses. Nothing here talks to a server; it's the# wire format, printed so the shapes are concrete before we start using the SDK.import json# 1. Request — has an id, so it expects a matching response back.initialize_request = {    "jsonrpc": "2.0",    "id": 1,    "method": "initialize",    "params": {        "protocolVersion": "2025-06-18",        "capabilities": {},        "clientInfo": {"name": "notes-client", "version": "1.0.0"},    },}# 2. Response — same id, and exactly one of "result" or "error".initialize_response = {    "jsonrpc": "2.0",    "id": 1,    "result": {        "protocolVersion": "2025-06-18",        "capabilities": {"tools": {}, "resources": {}, "prompts": {}},        "serverInfo": {"name": "DocumentMCP", "version": "1.0.0"},    },}# 3. Notification — no id, so no reply is expected or allowed.initialized_notification = {"jsonrpc": "2.0", "method": "notifications/initialized"}for label, message in [    ("REQUEST", initialize_request),    ("RESPONSE", initialize_response),    ("NOTIFICATION", initialized_notification),]:    print(f"--- {label} ---")    print(json.dumps(message, indent=2))    print()

## 1.3 MCP clientsA client is the half of the protocol that lives inside the host application. Its job is tolaunch or connect to one server, complete the handshake, and then translate the host's needsinto MCP calls.### Connection lifecycle```1. INITIALIZE     client → initialize request  (protocol version, client capabilities)                  server → initialize result   (protocol version, server capabilities)                  client → notifications/initialized                  ── nothing else is legal before this completes ──2. OPERATION      tools/list        resources/list       prompts/list                  tools/call        resources/read       prompts/get                  ← notifications/tools/list_changed, etc. (server-initiated)3. SHUTDOWN       close the transport; for stdio, terminate the subprocess```Capability negotiation during step 1 is what makes the protocol extensible: each side advertiseswhat it supports, and neither assumes features the other didn't declare.### Clients you'll meet| Client | What it's for ||---|---|| **MCP Inspector** | Browser debugging UI, launched by `uv run mcp dev` — used in §2.3 || **Claude Desktop / Claude Code** | Real hosts; configured via JSON config to spawn your server || **`mcp_client.py`** | The one written by hand in this course — §3.1 |The point of the protocol is that a server written for one of these works unchanged in all ofthem. You'll test `mcp_server.py` in the Inspector, then point a hand-written client at the verysame file without editing it.

---# 2 — Hands-on with MCP servers## 2.1 Project setupThe course uses **uv**, Astral's Rust-based Python package manager, in place ofpip/venv/poetry. (See the UV summary in `★ Introduction to MCP Course.ipynb` for the commandreference.) The commands that matter here:```bashuv sync                        # install exactly what uv.lock pinsuv run mcp dev mcp_server.py   # run the server with the Inspector UIuv run mcp run mcp_server.py   # run it bare, over stdiouv run main.py                 # run the finished CLI chat app```### DependenciesFrom `Examples/pyproject.toml`:| Package | Why ||---|---|| `mcp[cli]` | The protocol SDK. The `[cli]` extra is what provides the `mcp dev` command || `anthropic` | Claude API client, used by `core/claude.py` || `prompt-toolkit` | The CLI's autocomplete and key bindings (§3.5) || `python-dotenv` | Loads `ANTHROPIC_API_KEY` from `.env` |### Layout```Examples/├── mcp_server.py     # the server — tools, resources, prompts   (§2.2, 3.2, 3.4)├── mcp_client.py     # the client — connect + protocol wrappers (§3.1)├── main.py           # entry point: wires client → chat → CLI├── core/│   ├── claude.py     # thin Anthropic API wrapper│   ├── chat.py       # the agentic loop (tool_use → execute → resend)│   ├── cli_chat.py   # adds @mentions and /commands              (§3.3, 3.5)│   ├── cli.py        # prompt-toolkit UI and autocomplete         (§3.5)│   └── tools.py      # collects tools from N clients, routes calls└── pyproject.toml```Note the split: `mcp_server.py` and `mcp_client.py` are pure protocol; everything Claude-specificlives in `core/`. That separation is the point — the server has no idea a model exists.

## 2.2 Defining tools with MCP**Tools are model-controlled.** Claude reads their names, descriptions and JSON schemas, thendecides on its own when to invoke one. That makes the metadata part of your prompt engineering,not just documentation — a vague description is the single most common reason a model fails tocall a tool it should have.`FastMCP` handles the schema generation. A decorator plus type hints is enough:```pythonfrom mcp.server.fastmcp import FastMCPfrom pydantic import Fieldmcp = FastMCP("DocumentMCP", log_level="ERROR")@mcp.tool(    name="read_doc_contents",    description="Read the contents of a document and return it as a string.",)def read_document(doc_id: str = Field(description="Id of the document to read")):    ...```Three things are happening:- `name=` is the identifier Claude calls. It's independent of the Python function name —  `read_document` the function is `read_doc_contents` the tool.- `description=` is the model's only explanation of *when* to use it.- `Field(description=...)` per parameter documents each argument **inside the generated JSON  schema**, so the model knows what to put in each slot.Raising an exception (e.g. `ValueError` for a missing doc) is the correct failure path — the SDKconverts it into an error result the model can read and react to, rather than crashing the server.

In [ ]:
# A miniature version of Examples/mcp_server.py, built here so we can introspect it live.# Same decorators, same patterns, just fewer documents.from mcp.server.fastmcp import FastMCPfrom pydantic import Fielddemo = FastMCP("DemoDocs", log_level="ERROR")# Stand-in for a database or file system. Edits live only as long as this kernel.docs = {    "deposition.md": "This deposition covers the testimony of Angela Smith, P.E.",    "report.pdf": "The report details the state of a 20m condenser tower.",    "plan.md": "The plan outlines the steps for the project's implementation.",}# Tool that looks up a document by id and returns its raw contents.@demo.tool(    name="read_doc_contents",    description="Read the contents of a document and return it as a string.",)def read_document(doc_id: str = Field(description="Id of the document to read")):    if doc_id not in docs:        raise ValueError(f"Doc with id {doc_id} not found")    return docs[doc_id]# Tool that finds an exact substring in a document and replaces it with new text.@demo.tool(    name="edit_document",    description="Edit a document by replacing a string in the documents content with a new string",)def edit_document(    doc_id: str = Field(description="Id of the document that will be edited"),    old_str: str = Field(description="The text to replace. Must match exactly, including whitespace"),    new_str: str = Field(description="The new text to insert in place of the old text"),):    if doc_id not in docs:        raise ValueError(f"Doc with id {doc_id} not found")    docs[doc_id] = docs[doc_id].replace(old_str, new_str)print(f"Server {demo.name!r} defined with 2 tools.")

In [ ]:
# This is exactly what the client receives from a tools/list call, and what gets forwarded# to Claude as the tool definitions. Note the Field descriptions surfacing in the schema.import jsontools = await demo.list_tools()for tool in tools:    print(f"■ {tool.name}\n  {tool.description}")    print("  input_schema:")    for line in json.dumps(tool.inputSchema, indent=2).split("\n"):        print(f"    {line}")    print()

Two details worth noticing in that schema output:- Every `Field(description=...)` landed in `properties.<arg>.description`. That text travels all  the way to the model.- The schema's `title` is `read_documentArguments` — derived from the **Python function name**,  not the `name=` you gave the decorator. Harmless, but it's a good reminder that the two names  are genuinely independent.

In [ ]:
# Invoking a tool the way the protocol does. FastMCP.call_tool returns a list of content# blocks — the same TextContent objects that come back over the wire as tools/call results.result = await demo.call_tool("read_doc_contents", {"doc_id": "plan.md"})print("Raw result:", result)print("Text:", result[0].text)# Mutating tool: run it, then read the document back to confirm the change stuck.await demo.call_tool(    "edit_document",    {"doc_id": "plan.md", "old_str": "implementation", "new_str": "rollout"},)after_edit = await demo.call_tool("read_doc_contents", {"doc_id": "plan.md"})print("After edit:", after_edit[0].text)

In [ ]:
# Errors are values, not crashes. A failing tool returns an error result the model can read# and recover from, which is why raising ValueError inside a tool is the right move.from mcp.server.fastmcp.exceptions import ToolErrortry:    await demo.call_tool("read_doc_contents", {"doc_id": "does_not_exist.md"})except ToolError as err:    # Called directly, FastMCP raises. Over a real transport the SDK catches this and sends    # back {"isError": true, ...}, which core/tools.py turns into a tool_result block for Claude.    print(f"ToolError -> {err}")

## 2.3 The server inspectorAn MCP server has no UI and, over stdio, no visible output. The **Inspector** is the debuggingclient that fixes that:```bashuv run mcp dev mcp_server.py```That starts your server *and* a local web app connected to it. What you get:| Pane | Use ||---|---|| **Tools** | List every tool, fill in arguments from a generated form, run it, see the raw result || **Resources** | Browse direct resources and fill in templates by URI || **Prompts** | Render a prompt with arguments and read the exact messages produced || **Notifications** | Server log output and protocol-level events |The workflow the course pushes: **write a primitive → check it in the Inspector → only then wireit into a client.** It isolates "is my server correct?" from "is my client correct?", which areotherwise very easy to confuse.### The silence gotcha```bashuv run mcp run mcp_server.py   # appears to hang — no output, never returns```This is correct behaviour, not a bug. `transport="stdio"` means the server is sitting on stdinwaiting for JSON-RPC frames from a client that hasn't arrived. Use `mcp dev` when you wantsomething to look at.

## 2.4 Course satisfaction surveyAdministrative checkpoint — no technical content.

---# 3 — Connecting with MCP clients## 3.1 Implementing a client`Examples/mcp_client.py` wraps the SDK's session into a class that (a) spawns the server, (b)completes the handshake, and (c) exposes one method per protocol operation.The three imports that do the work:| Piece | Role ||---|---|| `StdioServerParameters` | *Describes* the subprocess to launch — command, args, env. Inert. || `stdio_client(params)` | Actually spawns it, yields its `(read, write)` pipes || `ClientSession(read, write)` | Speaks MCP over those pipes |### Connecting```pythonasync def connect(self):    server_params = StdioServerParameters(        command=self._command, args=self._args, env=self._env,    )    stdio_transport = await self._exit_stack.enter_async_context(        stdio_client(server_params)    )    _stdio, _write = stdio_transport    self._session = await self._exit_stack.enter_async_context(        ClientSession(_stdio, _write)    )    await self._session.initialize()   # ← the handshake; nothing works before this```**Why `AsyncExitStack`?** Both `stdio_client` and `ClientSession` are async context managers, andthey must be torn down in the reverse of their setup order. Nesting `async with` blocks wouldforce all your logic to live inside the nesting. The exit stack lets `connect()` and `cleanup()`be separate methods while still guaranteeing correct unwind order, even if setup fails halfway.`MCPClient` then implements `__aenter__`/`__aexit__` on top, so callers get the clean form:```pythonasync with MCPClient(command="uv", args=["run", "mcp_server.py"]) as client:    tools = await client.list_tools()```### The four operation wrappersEach is a thin pass-through to the session — the value is in the return types:| Method | Returns | Notes ||---|---|---|| `list_tools()` | `list[types.Tool]` | `.name`, `.description`, `.inputSchema` — exactly what Claude needs || `call_tool(name, input)` | `types.CallToolResult` | `input` is a plain dict matching the schema || `list_prompts()` | `list[types.Prompt]` | Each carries `.arguments` || `get_prompt(name, args)` | `list[PromptMessage]` | Template already filled in || `read_resource(uri)` | unwrapped Python value | See §3.3 |### Windows gotcha```pythonif sys.platform == "win32":    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())```The stdio transport depends on asyncio subprocess pipes, and on Windows only the Proactor eventloop implements them. Without this, `connect()` fails outright. This is at the bottom of`mcp_client.py` and is the reason the client can't simply be `await`-ed from a notebook cell —the notebook kernel owns the loop policy.

In [ ]:
# EXCERPT — this one can't execute in a notebook: the client sets its own event loop policy# and spawns a subprocess, both of which fight the kernel's already-running loop.# Save as Examples/session_demo.py and run:  cd Examples && uv run session_demo.py## import sys, asyncio# from mcp_client import MCPClient## async def main():#     async with MCPClient(command="uv", args=["run", "mcp_server.py"]) as client:#         # Tools — what Claude may call#         for tool in await client.list_tools():#             print(f"  tool:   {tool.name}: {tool.description}")##         # Resources — what the app fetches by URI#         doc_ids = await client.read_resource("docs://documents")#         print(f"  docs:   {doc_ids}")#         one_doc = await client.read_resource("docs://documents/plan.md")#         print(f"  plan:   {one_doc}")##         # Prompts — what the user triggers#         for prompt in await client.list_prompts():#             print(f"  prompt: /{prompt.name}")##         # Calling a tool. Note .content[0].text — CallToolResult wraps content blocks.#         result = await client.call_tool("read_doc_contents", {"doc_id": "plan.md"})#         print(f"  call:   {result.content[0].text}")## if __name__ == "__main__":#     # Required on Windows: only the Proactor loop implements subprocess pipes.#     if sys.platform == "win32":#         asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())#     asyncio.run(main())print("See Examples/mcp_client.py — its own main() does exactly this.")

## 3.2 Defining resources**Resources are application-controlled.** The model does not choose them; the client requeststhem by URI. They're read-only lookups — the `GET` to a tool's `POST`.Two kinds, distinguished only by whether the URI has a placeholder:```python# Direct — a fixed URI, no parameters@mcp.resource("docs://documents", mime_type="application/json")def list_docs() -> list[str]:    return list(docs.keys())# Templated (RFC 6570) — {doc_id} becomes a function parameter@mcp.resource("docs://documents/{doc_id}", mime_type="text/plain")def fetch_doc(doc_id: str) -> str:    if doc_id not in docs:        raise ValueError(f"Doc with id {doc_id} not found")    return docs[doc_id]```The scheme (`docs://`) is yours to invent — it just has to be a valid URI. The two are listed by*different* protocol calls: `resources/list` returns only direct resources, while`resources/templates/list` returns the templated ones. A templated resource can't appear in a flatlist because there's no finite set of URIs to enumerate.`mime_type` matters more than it looks — the client uses it to decide how to decode the payload,which is what makes the JSON auto-parsing in §3.3 possible.

In [ ]:
# Adding both resource kinds to our in-notebook server.# Direct resource that lists the ids of every document available.@demo.resource("docs://documents", mime_type="application/json")def list_docs() -> list[str]:    return list(docs.keys())# Templated resource returning one document's contents, addressed by id in the URI.@demo.resource("docs://documents/{doc_id}", mime_type="text/plain")def fetch_doc(doc_id: str) -> str:    if doc_id not in docs:        raise ValueError(f"Doc with id {doc_id} not found")    return docs[doc_id]# Two separate protocol calls — this is the part that surprises people.print("resources/list (direct only):")for resource in await demo.list_resources():    print(f"  {resource.uri}  [{resource.mimeType}]")print("\nresources/templates/list (templated only):")for template in await demo.list_resource_templates():    print(f"  {template.uriTemplate}  [{template.mimeType}]")

## 3.3 Accessing resourcesOn the client side, `read_resource` has to unwrap a nested response. From `mcp_client.py`:```pythonasync def read_resource(self, uri: str) -> Any:    result = await self.session().read_resource(AnyUrl(uri))    resource = result.contents[0]    if isinstance(resource, types.TextResourceContents):        if resource.mimeType == "application/json":            return json.loads(resource.text)        return resource.text```Four things going on:1. **`AnyUrl(uri)`** — pydantic validates it's a real URI before sending.2. **`.contents[0]`** — a resource may return several parts; this client uses the first.3. **`isinstance(..., TextResourceContents)`** — the other case is `BlobResourceContents`   (base64 binary), which this client doesn't handle.4. **The mime-type branch** — because the server declared `application/json`, the client decodes   it into a real Python `list` instead of handing back a string that merely looks like JSON.   This is `mime_type` earning its keep: callers get `["plan.md", ...]`, not `'["plan.md", ...]'`.### Where resources surface: `@`-mentions`core/cli_chat.py` turns resources into a user-facing feature. Type `@plan.md` in the CLI and:```pythonasync def _extract_resources(self, query: str) -> str:    mentions = [word[1:] for word in query.split() if word.startswith("@")]    doc_ids = await self.list_docs_ids()          # docs://documents    mentioned_docs = []    for doc_id in doc_ids:        if doc_id in mentions:            content = await self.get_doc_content(doc_id)   # docs://documents/{doc_id}            mentioned_docs.append((doc_id, content))    return "".join(        f'\n<document id="{doc_id}">\n{content}\n</document>\n'        for doc_id, content in mentioned_docs    )```The contents are injected into the prompt **before** Claude ever sees the query, wrapped in XMLtags. This is the whole point of the model/application split: the app decided what context wasrelevant, so the model doesn't have to spend a tool call fetching it. The system prompt even tellsClaude not to bother reading a doc that's already been inlined.

In [ ]:
# read_resource on the direct resource. FastMCP hands back ReadResourceContents objects# carrying the serialized content plus the mime type the client branches on.contents = list(await demo.read_resource("docs://documents"))part = contents[0]print(f"mime_type: {part.mime_type}")print(f"raw content: {part.content!r}")# The mime-type branch mcp_client.py performs, reproduced here.import jsondecoded = json.loads(part.content) if part.mime_type == "application/json" else part.contentprint(f"decoded:   {decoded!r}   ({type(decoded).__name__})")

In [ ]:
# The templated resource: {doc_id} is filled from the URI itself.for doc_id in decoded:    part = list(await demo.read_resource(f"docs://documents/{doc_id}"))[0]    print(f"{doc_id:<16} [{part.mime_type}]  {part.content}")

In [ ]:
# Reproducing the @mention extraction from core/cli_chat.py against our in-notebook server.# Note it returns a formatted context string, not raw documents.async def extract_resources(query: str) -> str:    mentions = [word[1:] for word in query.split() if word.startswith("@")]    doc_ids = json.loads(list(await demo.read_resource("docs://documents"))[0].content)    mentioned_docs = []    for doc_id in doc_ids:        if doc_id in mentions:            content = list(await demo.read_resource(f"docs://documents/{doc_id}"))[0].content            mentioned_docs.append((doc_id, content))    return "".join(        f'\n<document id="{doc_id}">\n{content}\n</document>\n'        for doc_id, content in mentioned_docs    )user_query = "Summarise @plan.md and compare it to @report.pdf"print(f"User typed: {user_query}\n")print("Context injected into the prompt before Claude sees it:")print(await extract_resources(user_query))

## 3.4 Defining prompts**Prompts are user-controlled.** The human picks one, usually from a slash-command menu. They'renot strings — they return a list of message objects, so a prompt can seed an entire multi-turnexchange:```pythonfrom mcp.server.fastmcp.prompts import base@mcp.prompt(    name="format",    description="Rewrites the contents of the document in Markdown format.",)def format_document(    doc_id: str = Field(description="Id of the document to format"),) -> list[base.Message]:    prompt = f"""    Your goal is to reformat a document to be written with markdown syntax.    The id of the document you need to reformat is:    <document_id>    {doc_id}    </document_id>    Add in headers, bullet points, tables, etc as necessary. ...    Use the 'edit_document' tool to edit the document. ...    """    return [base.UserMessage(prompt)]````base.UserMessage` / `base.AssistantMessage` construct the roles. The value of putting this on the**server** rather than in the client is that the prompt ships with the tools it depends on — the`format` prompt knows to reach for `edit_document` because the same server provides both. Whoeverconnects gets a workflow that's already been tuned, instead of having to invent the wording.Note how the two prompts in `mcp_server.py` differ: `format` instructs Claude to use the`edit_document` **tool**, while `summarize` tells it to call `read_doc_contents` first. Promptsorchestrate tools.

In [ ]:
# Prompts return message objects, not strings.from mcp.server.fastmcp.prompts import base# Builds a prompt asking Claude to produce a concise summary of the given document.@demo.prompt(name="summarize", description="Summarizes the contents of the document.")def summarize_document(    doc_id: str = Field(description="Id of the document to summarize"),) -> list[base.Message]:    prompt = f"""    Your goal is to summarize the contents of a document.    The id of the document you need to summarize is:    <document_id>    {doc_id}    </document_id>    Use the 'read_doc_contents' tool to fetch the document's contents before summarizing.    Keep the summary concise and don't change the meaning of the original document.    """    return [base.UserMessage(prompt)]# prompts/list — this is what populates the slash-command menu, arguments included.for prompt in await demo.list_prompts():    required_args = ", ".join(        f"{arg.name}{'' if arg.required else '?'}" for arg in (prompt.arguments or [])    )    print(f"/{prompt.name} <{required_args}>\n    {prompt.description}\n")

In [ ]:
# prompts/get — the template rendered with real arguments.rendered = await demo.get_prompt("summarize", {"doc_id": "plan.md"})print(f"description: {rendered.description}")print(f"messages:    {len(rendered.messages)}\n")for message in rendered.messages:    print(f"[{message.role}] {message.content.text.strip()}")

## 3.5 Prompts in the clientThe last step is turning server prompts into `/`-commands the user can actually type.### Dispatch`core/cli_chat.py` intercepts anything starting with `/` before it becomes a normal query:```pythonasync def _process_command(self, query: str) -> bool:    if not query.startswith("/"):        return False    words = query.split()    command = words[0].replace("/", "")    messages = await self.doc_client.get_prompt(command, {"doc_id": words[1]})    self.messages += convert_prompt_messages_to_message_params(messages)    return True```So `/summarize plan.md` fetches the rendered prompt from the server and appends it straight to theconversation history. `_process_query` checks this first and returns early if it handled a command.### The type mismatchMCP's `PromptMessage` and Anthropic's `MessageParam` are different types from different SDKs, so`convert_prompt_message_to_message_param` bridges them — mapping the role, then pulling `.text` outof the content block (handling both single blocks and lists). This is the seam between "protocol"and "this particular model provider", and it's exactly the kind of glue the M+N architecture issupposed to confine to one place.### Autocomplete`core/cli.py` uses `prompt-toolkit` to make both primitives discoverable, driven entirely by whatthe server reported:| Trigger | Completions from | Source call ||---|---|---|| `/` | Prompt names + descriptions | `prompts/list` || `@` | Document ids | `resources/read` on `docs://documents` || space after `/cmd` | Document ids as the argument | same |`CliApp.initialize()` calls `refresh_resources()` and `refresh_prompts()` at startup to populatethem. A `CommandAutoSuggest` additionally ghosts the expected argument name (`prompt.arguments[0].name`)once you've typed a valid command — which works because `prompts/list` reports each prompt'sargument metadata.Notice that none of this UI is hardcoded to the document server. Point it at a different MCP serverand the menus repopulate themselves.

In [ ]:
# The PromptMessage -> MessageParam conversion, the seam between MCP and the Anthropic SDK.# Simplified from core/cli_chat.py: the real one also handles list-valued content.def convert_prompt_message_to_message_param(prompt_message):    role = "user" if prompt_message.role == "user" else "assistant"    content = prompt_message.content    if getattr(content, "type", None) == "text":        return {"role": role, "content": content.text}    if isinstance(content, list):        text_blocks = [            {"type": "text", "text": item.text}            for item in content            if getattr(item, "type", None) == "text"        ]        if text_blocks:            return {"role": role, "content": text_blocks}    return {"role": role, "content": ""}# Simulating what happens when the user types "/summarize plan.md".typed = "/summarize plan.md"words = typed.split()command, doc_argument = words[0].lstrip("/"), words[1]prompt_result = await demo.get_prompt(command, {"doc_id": doc_argument})messages = [convert_prompt_message_to_message_param(m) for m in prompt_result.messages]print(f"User typed: {typed}\n")print("Appended to conversation history, ready for the Anthropic Messages API:")print(json.dumps(messages, indent=2)[:600] + " ...")

---# 4 — Assessment and wrap up## 4.1 Final assessment — the facts worth retaining**Primitives and control**| | Tools | Resources | Prompts ||---|---|---|---|| Controlled by | Model | Application | User || Decorator | `@mcp.tool()` | `@mcp.resource(uri)` | `@mcp.prompt()` || Protocol calls | `tools/list`, `tools/call` | `resources/list`, `resources/templates/list`, `resources/read` | `prompts/list`, `prompts/get` || Returns | Any serialisable value | Text or blob + mime type | `list[base.Message]` || Surfaces as | Claude calling it mid-answer | `@mention` | `/command` |**Architecture**- Host contains clients; **one client per server**, always 1:1.- Servers are passive — they respond, never initiate (except notifications).- Transports: **stdio** (local subprocess) and **streamable HTTP** (remote).- Data layer is **JSON-RPC 2.0**; `initialize` must complete before anything else.- MCP turns **M×N** integrations into **M+N**.**Traps**- Direct and templated resources come from **two different list calls**.- The tool `name=` and the Python function name are independent.- Never `print()` to stdout in a stdio server — it corrupts the JSON-RPC stream.- `uv run mcp run server.py` producing no output is correct, not a hang.- On Windows, the client needs `WindowsProactorEventLoopPolicy`.## 4.2 MCP review### How a single query moves through the whole system```user: "Summarise @plan.md"  │  ├─ CliApp            reads input, autocompletes @ from docs://documents  ├─ CliChat           _extract_resources() → resources/read → inlines <document> into prompt  ├─ Chat.run()        sends messages + tools to Claude  │                      tools gathered by ToolManager.get_all_tools() → tools/list on every client  ├─ Claude            replies, possibly stop_reason="tool_use"  ├─ ToolManager       finds the client owning that tool → tools/call → returns tool_result blocks  ├─ Chat.run()        loops: resends history until stop_reason is not tool_use  └─ final text answer````core/chat.py` is that loop, and it's short — the agentic behaviour is just "while the model asksfor a tool, run it and hand the result back."### The design lessonThe server knows nothing about Claude. It exposes documents over a standard protocol; the hostdecides what to do with them. That's why `ToolManager._find_client_with_tool` can search acrossseveral servers at once, and why the CLI's autocomplete repopulates itself from whatever server it'spointed at. Swap the document server for a database server and none of `core/` changes.### Where to go next- **Sampling** — let the server request a model completion from the host.- **Roots** — let the client tell the server which directories are in scope.- **Elicitation** — let the server ask the user a question mid-operation.- **Streamable HTTP + auth** — the path from a local subprocess to a shared, remote server.- **Publishing** — register a server so other hosts can install it.

---## Appendix — gotchas hit while building this| Symptom | Cause | Fix ||---|---|---|| `uv run mcp run mcp_server.py` prints nothing | Correct: stdio server waiting for a client | Use `uv run mcp dev` for the Inspector || Client hangs or errors on connect (Windows) | Default event loop can't do subprocess pipes | `asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())` || Templated resource missing from the list | `resources/list` excludes templates by design | Call `resources/templates/list` too || Resource returns a JSON-looking string | Client didn't branch on mime type | Declare `mime_type="application/json"`, decode in the client || Claude never calls a tool it should | Description too vague | Rewrite `description=` and the per-arg `Field(description=...)` || Garbled protocol errors over stdio | Something wrote to stdout | Keep stdout clean; `log_level="ERROR"` |### Reference- Course: <https://anthropic-partners.skilljar.com/introduction-to-model-context-protocol>- Spec: <https://modelcontextprotocol.io/>- Working code: [`Examples/`](Examples/) — `mcp_server.py`, `mcp_client.py`, `core/`